In [ ]:
import os
import json
import pandas as pd

In [ ]:
data = pd.read_csv('/Users/vrishab/Downloads/NYC_Restaurant_Inspection_Results.csv', low_memory=False)

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
important_cols = ['CAMIS', 'DBA', 'BORO', 'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE', 'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE', 'INSPECTION TYPE']

In [ ]:
df = data[important_cols].copy()

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ','_')

In [ ]:
df

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_percent

In [ ]:
df = df.drop(columns=['grade'])
df.columns

In [ ]:
df = df.dropna(subset=['violation_code', 'violation_description'])
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df

In [ ]:
df.dtypes

In [ ]:
df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
df.dtypes

In [ ]:
df.sample(10)

In [ ]:
df['inspection_date'].min()

In [ ]:
text_cols = ['dba', 'boro', 'cuisine_description', 'action', 'violation_code', 'violation_description', 'critical_flag', 'inspection_type']
for col in text_cols:
    df[col] = df[col].str.strip()

In [ ]:
df['critical_flag'] = df['critical_flag'].str.lower()
df['violation_code'] = df['violation_code'].str.lower()

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

In [ ]:
df[df['inspection_date'] < '2000-01-01']

In [ ]:
df['score'].describe()

In [ ]:
df['critical_flag'].value_counts()

In [ ]:
df['violation_description'].sample(10)

In [ ]:
df['violation_code'].sample(10)

In [ ]:
critical = df[df['critical_flag'] == 'critical']

In [ ]:
top_critical_violations = critical['violation_description'].value_counts().head(10)
top_critical_violations

In [ ]:
def violations(df, n=10):
    return df['violation_description'].value_counts().head(n)

In [ ]:
critical_violations = violations(critical)
critical_violations

In [ ]:
import matplotlib.pyplot as plt
critical_violations.sort_values().plot(kind='barh', figsize=(10, 5))
plt.title('Top Critical Restaurant Violations')
plt.xlabel('Number of Violations')
plt.ylabel('Violation Description')
plt.tight_layout()
plt.show()

In [ ]:
def run_analysis(file_path):
    
    data = pd.read_csv(file_path, low_memory=False)

    important_cols = ['CAMIS', 'DBA', 'BORO', 'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE', 'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE', 'INSPECTION TYPE']

    df = data[important_cols].copy()

    df.columns = df.columns.str.lower().str.replace(' ','_')

    df = df.drop(columns=['grade'], errors='ignore')

    df = df.dropna(subset=['violation_code', 'violation_description'])

    df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')
    df['score'] = pd.to_numeric(df['score'], errors='coerce')

    text_cols = ['dba', 'boro', 'cuisine_description', 'action', 'violation_code', 'violation_description', 'critical_flag', 'inspection_type']
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()

    df['critical_flag'] = df['critical_flag'].str.lower()
    df['violation_code'] = df['violation_code'].str.lower()

    df = df.drop_duplicates()

    critical = df[df['critical_flag'] == 'critical']

    top_critical_violations = critical['violation_description'].value_counts().head(10)

    return top_critical_violations

In [ ]:
top_violations = run_analysis('/Users/vrishab/Downloads/NYC_Restaurant_Inspection_Results.csv')
top_violations

In [ ]:
import sys
!{sys.executable} -m pip install openai

In [ ]:
from openai import OpenAI

In [ ]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [ ]:
def generate_recommendations(top_violations):
    
    violations_text = '\n'.join([f'{v} ({c})' for v, c in top_violations.items()])
    
    prompt = f'''
    You are a food safety expert.

    Based on the following most common critical restaurant violations:
    
    {violations_text}
    
    Provide clear and practical recommendations for restaurants to avoid these violations.
    Keep it concise and professional.
    '''

    response = client.chat.completions.create(model='gpt-4.1-mini', messages=[{'role': 'user', 'content': prompt}])

    return response.choices[0].message.content

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'generate_recommendations',
            'description': 'Generates food safety recommendations based on the top critical violations found in the dataset.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'violations_summary': {
                        'type': 'string',
                        'description': 'A summary of the top critical violations to base recommendations on.'
                    }
                },
                'required': ['violations_summary']
            }
        }
    }
]

In [ ]:
def run_ai_agent(file_path, query='What should restaurants do to avoid critical violations?'):

    top_violations = run_analysis(file_path)
    violations_text = '\n'.join([f'{v} ({c})' for v, c in top_violations.items()])

    messages = [
        {'role': 'system', 'content': 'You are a food safety analyst agent. Use the available tool to help answer the users question.'},
        {'role': 'user', 'content': f'{query}\n\nHere are the top critical violations found:\n{violations_text}'}
    ]

    response = client.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        tools=tools,
        tool_choice='auto'
    )

    msg = response.choices[0].message

    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        recommendations = generate_recommendations(top_violations)

        return {
            'top_violations': top_violations,
            'recommendations': recommendations
        }
    else:
        return {
            'top_violations': top_violations,
            'recommendations': msg.content
        }

In [ ]:
result = run_ai_agent('/Users/vrishab/Downloads/NYC_Restaurant_Inspection_Results.csv')

result['top_violations']
result['recommendations']